## Heart Disease Prediction with Ensemble Learning

### 1. Introduction

This Jupyter Notebook implements an ensemble learning approach to predict heart disease presence from a tabular dataset. The primary goal is to train an `EnsembleClassifier` using the `likelihood` library, evaluate its performance on test data, and generate a submission file for a prediction task. The notebook demonstrates how to build, train, and utilize an ensemble model for classification problems.

### 2. Methodology

The methodology employed in this notebook consists of several key steps:

1.  **Data Loading & Preprocessing:**
    *   Loads training data (`train.csv`) and test data (`test.csv`) using pandas.
    *   Preprocesses the data, including:
        *   Converting the 'Sex' column to a categorical type.
        *   Replacing string values in the 'Heart Disease' column with numerical representations (1 for presence, 0 for absence).
2.  **Pipeline Creation:** A `Pipeline` object is created from an `ensemble_config.json` file, defining the sequence of transformations applied to the data – specifically, a model fitting process.
3.  **Model Training & Fitting:** The `EnsembleClassifier` is initialized and trained on the training data using the defined pipeline. A validation split (20%) is incorporated to monitor performance during training.
4.  **Test Data Transformation:** The test data is transformed using the same pipeline that was used for training, ensuring consistency in feature engineering.
5.  **Prediction Generation:** Predictions are generated on the transformed test data using the trained `EnsembleClassifier`. Probabilities are also calculated.
6.  **Model Evaluation:** Individual models within the ensemble are evaluated by printing their F1-score and validation loss. This provides insights into the performance of each model component.
7. **Submission File Generation**: A submission file (`sample_submission.csv`) is created containing predicted probabilities for the 'Heart Disease' target variable based on the final predictions from the ensemble.

In [1]:
%%capture
import sys

# Añade el directorio principal al path de búsqueda para importar módulos desde esa ubicación
sys.path.insert(0, "..")

# Desactivar los warnings para evitar mensajes innecesarios durante la ejecución
import warnings

import math
import numpy as np
import pandas as pd
from likelihood.models.ensemble import EnsembleClassifier
from likelihood import Pipeline

In [2]:
df = pd.read_csv("train.csv")
df["Heart Disease"] = df["Heart Disease"].replace({"Presence": 1, "Absence": 0})
df["Sex"] = df["Sex"].astype("category")
etl_pipe = Pipeline("ensemble_config.json")
x_train, y_train, importances = etl_pipe.fit(df.copy().drop(columns=["id"]))
X_train = np.asarray(x_train.to_numpy()).astype(np.float32)
y_train = y_train.reshape((y_train.size, 1))
_train = (np.eye(y_train.max() + 1)[y_train]).reshape((-1, 2))
y_train = np.asarray(_train).astype(np.float32)

df_test = pd.read_csv("test.csv")
df_test["Sex"] = df_test["Sex"].astype("category")
X_test = etl_pipe.transform(df_test.copy().drop(columns=["id"]))
X_test.insert(0, "id", df_test["id"])
X_test = np.asarray(X_test.drop(columns=["id"]).to_numpy()).astype(np.float32)

In [3]:
# Define parameter ranges for variation
param_ranges = {
    "units": (10, 20),
    "activation": ["selu", "relu"],
    "num_layers": (1, 5),
    "dropout": (0.0, 0.5),
}

# Create and train the ensemble
ensemble = EnsembleClassifier(
    n_models=2, param_ranges=param_ranges, seed_range=(0, 100), voting_method="soft", verbose=1
)

ensemble.fit(X_train, y_train, epochs=1, validation_split=0.2)
ensemble.save("./ensemble")
ensemble = EnsembleClassifier.load("./ensemble")

# Predictions
predictions = ensemble.predict(X_test)
probabilities = ensemble.predict_proba(X_test)

# Evaluate individual models
scores = ensemble.get_model_scores()
for score in scores:
    print(
        f"Model {score['model_id']}: F1={score['f1_score']:.3f}, Val Loss={score['val_loss']:.4f}"
    )

Training model 1/2...
Training model 2/2...
Ensemble trained with 2 models.
Model 1: F1=0.845, Val Loss=0.3362
Model 2: F1=0.842, Val Loss=0.3604


In [4]:
pred = ensemble.predict_proba(X_test)

df = pd.DataFrame(columns=["id", "Heart Disease"])
df["id"] = df_test["id"]
df["Heart Disease"] = pred[:, 1]
# truncate 1 decimal places
df["Heart Disease"] = df["Heart Disease"].apply(lambda x: float(math.floor(x * 10) / 10))

df.to_csv("sample_submission.csv", index=False)

In [5]:
ensemble.fit(X_train, y_train, epochs=1, validation_split=0.2)

# Evaluate individual models
scores = ensemble.get_model_scores()
for score in scores:
    print(
        f"Model {score['model_id']}: F1={score['f1_score']:.3f}, Val Loss={score['val_loss']:.4f}"
    )

Training model 1/2...
Training model 2/2...
Ensemble trained with 2 models.
Model 1: F1=0.855, Val Loss=0.3694
Model 2: F1=0.797, Val Loss=0.3046


### 3. Analysis and Results

The notebook utilizes an `EnsembleClassifier` to achieve improved prediction accuracy compared to a single model. The following table summarizes the key results obtained during the evaluation process:

| Model ID | F1-Score   | Val Loss     |
| :------- | :--------- | :----------- |
|  *See Output* | *See Output* | *See Output* |

**Note:** The actual F1-score and validation loss values will be printed to the console during execution. These values represent the performance of each individual model within the ensemble, as determined by the `get_model_scores()` function. The final prediction probabilities are then used to generate the submission file.

### 4. Conclusions

The implementation of an ensemble learning approach using the `EnsembleClassifier` demonstrates a viable strategy for predicting heart disease presence from tabular data. The model achieved promising results, as evidenced by the F1-scores and validation losses reported during evaluation. Further improvements could be explored through techniques such as increasing the number of epochs in training, tuning the parameters within the `ensemble_config.json` file (e.g., exploring different activation functions or dropout rates), or incorporating more sophisticated voting methods. The generated submission file provides a prediction ready for evaluation against the ground truth.